# 01 — Explore the Podlog DB

This notebook is the starting point for ad-hoc DB exploration on Podlog (Issue #607).

**Workflow.** Each cell is self-contained — run from top to bottom on first use. The connection string is read from the `DATABASE_URL` env var injected by the `explore` Docker service, so `pd.read_sql(...)` works with no setup.

**Where to put your own work.** Make a new `.ipynb` next to this one (or in `/workspace/` directly). Anything outside `notebooks/examples/` is gitignored, so your scratch work doesn't pollute the repo.

**Schema reference.** The cell below dumps the table list with column names + types. Treat it as a living cheat sheet — re-run after a migration.

## 1. Connect to the DB

In [ ]:
import os
from sqlalchemy import create_engine, inspect
import pandas as pd

engine = create_engine(os.environ["DATABASE_URL"])
engine

## 2. Schema cheat sheet

Tables, columns, and types. Run this once and keep the output around when writing queries.

In [ ]:
insp = inspect(engine)
rows = []
for table in sorted(insp.get_table_names()):
    for col in insp.get_columns(table):
        rows.append({
            "table": table,
            "column": col["name"],
            "type": str(col["type"]),
            "nullable": col["nullable"],
        })
schema = pd.DataFrame(rows)
schema

## 3. Sample queries

A few starter queries to demonstrate the pattern. Copy/adapt these in your own notebooks.

In [ ]:
# Episodes per feed (top 20)
feeds_by_count = pd.read_sql(
    """
    SELECT f.title AS feed, COUNT(e.id) AS episodes
    FROM feeds f
    LEFT JOIN episodes e ON e.feed_id = f.id
    GROUP BY f.title
    ORDER BY episodes DESC
    LIMIT 20
    """,
    engine,
)
feeds_by_count

In [ ]:
# Segments per status — quick health check
status = pd.read_sql(
    """
    SELECT status, COUNT(*) AS episodes
    FROM episodes
    GROUP BY status
    ORDER BY episodes DESC
    """,
    engine,
)
status

In [ ]:
# Recent episodes (last 10) with duration in minutes
recent = pd.read_sql(
    """
    SELECT title, duration_secs / 60.0 AS minutes, status, published_at
    FROM episodes
    ORDER BY published_at DESC NULLS LAST
    LIMIT 10
    """,
    engine,
)
recent

## 4. Plot something

Plotly renders inline. Bar chart of episodes per feed using the DataFrame above.

In [ ]:
import plotly.express as px

fig = px.bar(
    feeds_by_count,
    x="feed",
    y="episodes",
    title="Episodes per feed",
)
fig.update_layout(xaxis_tickangle=-30)
fig.show()

## What next

- Spin up your own notebooks alongside this one — they're persisted on the host at `notebooks/` and gitignored except for `examples/`.
- The `engine` you built in cell 1 is plain SQLAlchemy — use `pd.read_sql_query(text(...), engine, params=...)` if you want bind parameters.
- See `docs/guide/17-explore.md` for the full guide and operational notes (logs, restart, port conflicts).